In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.functional as F
import torch.utils.data as data

import evaluation

In [13]:
target_discrete_intervals = torch.tensor([100000, 350000]) # 0 to mniej niż 100k$, 1 pomiędzy, 2 więcej
validation_percent = 0.3 # Ile w walidacyjnym

categorical_columns = ["HallwayType", "HeatingType", "AptManageType", "TimeToBusStop", "TimeToSubway", "SubwayStation"]
columns_to_drop = []

In [14]:
def Get_Train_Data():
    train_data = pd.read_csv("train_data.csv")
    train_data = train_data.drop(columns=columns_to_drop)
    size = len(train_data)

    train_categorical = pd.get_dummies(train_data[categorical_columns]).astype(int)

    train_numerical = train_data.drop(columns=categorical_columns)
    return train_numerical, train_categorical, size

In [15]:
def Get_Final_Test_Data():
    final_test_data = pd.read_csv("test_data.csv", index_col=None)
    final_test_data = final_test_data.drop(columns=columns_to_drop)
    size = len(final_test_data)

    final_test_categorical = pd.get_dummies(final_test_data[categorical_columns]).astype(int)

    final_test_numerical = final_test_data.drop(columns=categorical_columns)
    return final_test_numerical, final_test_categorical, size

In [24]:
def Get_Train_And_Validate_Loaders():
    train_numerical, train_categorical, size = Get_Train_Data()
    train_indices = np.random.rand(size)>validation_percent
    print("Train data size: ", size)

    numerical_data = torch.from_numpy(train_numerical.values[train_indices,1:]).float()
    categorical_data = torch.from_numpy(train_categorical.values[train_indices]).float()
    numerical_train_targets = torch.from_numpy(train_numerical.values[train_indices, 0]).float()
    discrete_train_targets = torch.bucketize(numerical_train_targets, target_discrete_intervals) # Dyskretne etykiety

    validate_numerical_data = torch.from_numpy(train_numerical.values[~train_indices, 0:]).float()
    validate_categorical_data = torch.from_numpy(train_categorical.values[~train_indices]).float()
    numerical_validate_targets = torch.from_numpy(train_numerical.values[~train_indices, 0]).float()
    discrete_validate_targets = torch.bucketize(numerical_validate_targets, target_discrete_intervals) # Dyskretne etykiety

    train_loader = data.TensorDataset(numerical_data, categorical_data, discrete_train_targets)
    validate_loader = data.TensorDataset(validate_numerical_data, validate_categorical_data, discrete_validate_targets)

    return train_loader, validate_loader

In [22]:
def Get_Final_Test_Loader():
    final_test_numerical, final_test_categorical, size = Get_Final_Test_Data()
    print("Final test data size: ", size)

    final_test_numerical_data = torch.from_numpy(final_test_numerical.values).float()
    final_test_categorical_data = torch.from_numpy(final_test_categorical.values).float()

    final_test_loader = data.TensorDataset(final_test_numerical_data, final_test_categorical_data)

    return final_test_loader

In [26]:
train_loader, validate_loader = Get_Train_And_Validate_Loaders()
final_test_loader = Get_Final_Test_Loader()

Train data size:  4124
Final test data size:  1767


In [ ]:
def Get_Accuracy(model, data_loader):
    targets = []
    preds = []
    model.eval() # <------------------------ trzeba pamiętać żeby dać później model.train()
    for x, cat_x, labels in data_loader:
        x, cat_x, labels = x.to(device), cat_x.to(device), labels.to(device)
        output = model(x, cat_x)
        output = 0
        preds.append(output)
        targets.append(labels)

    accuracy = evaluation.calc_accuracy(preds, targets)
    return accuracy

In [ ]:
def Save_Final_Predictions_To_File(model, final_test_loader):
    model.eval()
    output_array = []
    for x, cat_x in final_test_loader:
        x, cat_x = x.to(device), cat_x.to(device)
        out = model(x, cat_x)
        output_array.append(out)

    pd.Series(output_array).to_csv("pred.csv", index=False, header=False)
